# 01 · Data Preprocessing & Causal Feature Engineering

**Objective.** Load the TxSON33 hourly dataset, document its schema and the target
(`SWC_20`, 20 cm soil moisture), and demonstrate the **causal** feature engineering that
every later experiment depends on — forward-fill and trailing windows only, never a
backward fill, so no feature at hour *t* can see the future.

**Provenance & reproducibility**

- **Results shown here:** 🔄 **computed fresh in this notebook.** Nothing here is loaded from the November-17 result CSVs; it loads the raw parquet and engineers features live.
- **Experimental setup:** Raw file has all 33 TxSON stations and years 2014–2026; the study uses the 4 complete-record stations (FD11, FD02, CB06, CB25) over 2015–2024. Target `SWC_20`.
- **Implementation:** `src.data_processing.load_dataframe`, `build_station_year_store`; `src.features.engineer_causal` (see `src/features.py` for the leakage-safe rules).
- **Original code:** this is a data/feature-engineering demo; it calls the same `src/` functions the training notebooks use. The original monolithic notebook is archived at `archive/cnn_experiments_original_monolithic.ipynb`.
- **Reproduce:** Just run the cells (CPU, ~30 s). The dataset auto-downloads from the public GitHub mirror (`src.config.DATA_URL`) if it is not found locally.

In [1]:
# --- project setup: portable across local / Google Drive / fresh Colab ---
import os, sys

def find_project_root(marker="src"):
    # 1) walk up from the current directory (local checkout or Drive upload)
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    # 2) common Colab upload locations
    for c in ("/content/soil_moisture_research",
              "/content/drive/MyDrive/soil_moisture_research"):
        if os.path.isdir(os.path.join(c, marker)):
            return c
    raise FileNotFoundError(
        "Could not find the project root (a folder containing 'src/').\n"
        "Upload the whole soil_moisture_research/ folder to Drive/Colab and "
        "open this notebook from inside notebooks/.")

ROOT = find_project_root()
sys.path.insert(0, ROOT)
os.chdir(ROOT)
print("project root set ->", os.path.basename(ROOT))  # portable: no absolute path

project root set -> soil_moisture_research


In [2]:
import pandas as pd
from src.data_processing import load_dataframe, build_station_year_store
from src import config as C
df = load_dataframe()
print('rows:', len(df), '| stations:', df.Station.nunique(), '| years:', sorted(df.year.unique()))
df[['Station','Timestamp','SWC_5','SWC_20','Ppt','Tair','RH','T_20']].head()

rows: 2720775 | stations: 33 | years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


,Station,Timestamp,SWC_5,SWC_20,Ppt,Tair,RH,T_20
0,CB01,2014-09-29 18:00:00,0.080,0.072,7.87,29.260,35.380,30.73
1,CB01,2014-09-29 19:00:00,0.079,0.073,0.00,26.635,46.690,30.92
2,CB01,2014-09-29 20:00:00,0.078,0.073,0.00,24.190,49.935,30.92
3,CB01,2014-09-29 21:00:00,0.077,0.073,0.00,22.310,56.460,30.79
4,CB01,2014-09-29 22:00:00,0.076,0.072,0.00,21.840,61.575,30.52


### Causal feature engineering (one station-year)
`engineer_causal` adds log / trailing-accumulated / lagged rainfall, forward-filled
meteorology, wind vectors, and day-of-year encodings. Every column is a function of the
present or past only.

In [3]:
store = build_station_year_store(df, C.SELECTED_STATIONS)
g = store[('FD11', 2020)]
cols = ['Timestamp','SWC_20','Ppt','Ppt_log','Ppt_24h','Ppt_log_lag1','hours_since_rain','Tair','sin_doy']
print('engineered (station,year) frames:', len(store))
g[cols].head()

engineered (station,year) frames: 40


,Timestamp,SWC_20,Ppt,Ppt_log,Ppt_24h,Ppt_log_lag1,hours_since_rain,Tair,sin_doy
0,2020-01-01 00:00:00,0.277,0.0,0.0,0.0,0.0,0,10.05,0.017202
1,2020-01-01 01:00:00,0.277,0.0,0.0,0.0,0.0,1,9.81,0.017202
2,2020-01-01 02:00:00,0.277,0.0,0.0,0.0,0.0,2,9.61,0.017202
3,2020-01-01 03:00:00,0.277,0.0,0.0,0.0,0.0,3,9.35,0.017202
4,2020-01-01 04:00:00,0.277,0.0,0.0,0.0,0.0,4,9.11,0.017202


The processed `(station, year)` store is what notebooks **02–07** consume. It is rebuilt
on demand (cheap, a few seconds), so no large intermediate file needs to be committed.